# 개방ID `8463049173` 수동 검토

## tl;dr

이 ID는 **성균관대학교 무역대학원** 계열의 폐교·폐과 잔존 ID로 판단된다. 2025년 9개 패널·49행의 실질 측정값은 모두 0이고 `국제금융학과`는 2022~2025년 모두 `폐과`다. KEDI 원본에서도 성균관대학교 무역대학원이 2022~2024년 `폐교`로 직접 확인된다.

## Context & Methods

EDSS 통합 DuckDB를 읽기 전용으로 조회하고 EDSS–KEDI 신원표와 행 매칭 증거표를 확인했다. KEDI 원본 XLSX 내부 XML에서 2022~2024년 학교 행을 직접 추출해 학교명·지역·주야간·폐교 상태를 대조했다.

### Key Assumptions

`0`은 결측치가 아니라 명시적 값으로 취급한다. 학교명 직접키가 없는 EDSS 자료이므로 학교 후보가 아닌 원본 ID를 보존하며 자동 조인은 하지 않는다.

## Data


In [1]:
import csv
import decimal
import os
import re
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

import duckdb

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
database_path = Path(os.environ.get(
    'EDSS_DUCKDB_PATH',
    '/Users/joocheol/Documents/GitHub/edss/data/processed/edss/restricted/edss_all.duckdb',
))
identity_path = repo_root / 'data/processed/edss_0101_kedi_openid_identity_2009_2025.csv'
evidence_path = repo_root / 'data/processed/edss_0101_kedi_row_match_evidence_2009_2025.csv'
kedi_dir = repo_root / 'data/raw/kedi/higher_education_school'
assert database_path.exists() and identity_path.exists() and evidence_path.exists() and kedi_dir.exists()
connection = duckdb.connect(str(database_path), read_only=True)
open_id = '8463049173'
duckdb.__version__, database_path.name

('1.4.1', 'edss_all.duckdb')

## Results

### 1. 자동 신원표와 2025년 패널 범위


In [2]:
with identity_path.open(encoding='utf-8-sig', newline='') as handle:
    identity_row = next(row for row in csv.DictReader(handle) if row['openid'] == open_id)
with evidence_path.open(encoding='utf-8-sig', newline='') as handle:
    candidate_rows = [row for row in csv.DictReader(handle) if row['openid'] == open_id]
assert identity_row['first_edss_year'] == '2022'
assert identity_row['last_edss_year'] == '2025'
assert identity_row['edss_year_count'] == '4'
assert identity_row['identity_status'] == 'unmatched'
assert len(candidate_rows) == 1
assert candidate_rows[0]['kedi_school_name'] == '성균관대학교 무역대학원'
assert candidate_rows[0]['kedi_school_status'] == '폐교'
identity_row, {key: candidate_rows[0][key] for key in ['year', 'kedi_school_name', 'kedi_school_status', 'match_families']}

({'openid': '8463049173',
  'first_edss_year': '2022',
  'last_edss_year': '2025',
  'edss_year_count': '4',
  'direct_match_year_count': '0',
  'latest_direct_school_name': '',
  'kedi_school_code_2025': '',
  'distinct_normalized_name_count': '0',
  'name_history': '',
  'identity_status': 'unmatched'},
 {'year': '2022',
  'kedi_school_name': '성균관대학교 무역대학원',
  'kedi_school_status': '폐교',
  'match_families': 'graduates'})

In [3]:
tables = connection.execute("""
SELECT table_schema, table_name FROM information_schema.columns
WHERE column_name='개방ID'
  AND table_schema IN ('higher_education', 'university_disclosure')
ORDER BY table_schema, table_name
""").fetchall()
panel_rows = []
for schema_name, table_name in tables:
    row_count = connection.execute(
        f"SELECT COUNT(*) FROM {schema_name}.{table_name} WHERE 개방ID=? AND 조사년도='2025'",
        [open_id],
    ).fetchone()[0]
    if row_count:
        panel_rows.append((schema_name, table_name, row_count))
assert len(panel_rows) == 9
assert sum(row[2] for row in panel_rows) == 49
panel_rows

[('higher_education', 'panel_0101', 1),
 ('higher_education', 'panel_0104', 1),
 ('higher_education', 'panel_0105', 1),
 ('higher_education', 'panel_0231', 2),
 ('higher_education', 'panel_0246', 12),
 ('university_disclosure', 'panel_0306', 3),
 ('university_disclosure', 'panel_0308', 5),
 ('university_disclosure', 'panel_0715', 18),
 ('university_disclosure', 'panel_1017', 6)]

### 2. 2025년 측정값과 학과 상태


In [4]:
dimension_columns = {
    '조사년도', '개방ID', '적용년도', '학기구분명', '수업연한명', '개설기간명',
    '학년명', '성별명', '학위과정구분명', '학과명', '학과한글명', '학과상태명',
    '단과대학명', '교육부계열명', '학과계열구분명', '주야간계절구분명',
    '본분교명', '시도명', '지역명', '학교구분명', '학제유형명', '교원구분명',
}
nonzero_measure_values = []
for schema_name, table_name, _ in panel_rows:
    cursor = connection.execute(
        f"SELECT * FROM {schema_name}.{table_name} WHERE 개방ID=? AND 조사년도='2025'",
        [open_id],
    )
    rows = cursor.fetchall()
    columns = [item[0] for item in cursor.description]
    for column_index, column_name in enumerate(columns):
        if column_name.startswith('_') or column_name in dimension_columns:
            continue
        for value in {str(row[column_index]) for row in rows if row[column_index] not in (None, '')}:
            try:
                if decimal.Decimal(value.replace(',', '')) != 0:
                    nonzero_measure_values.append((schema_name, table_name, column_name, value))
            except decimal.InvalidOperation:
                pass
assert nonzero_measure_values == []
nonzero_measure_values

[]

In [5]:
yearly_rows = connection.execute("""
SELECT 조사년도, 지역명, 본분교명, 고등교육학교_입학생수,
       고등교육학교_졸업생수, 고등교육학교_교원수,
       고등교육학교_재적학생수, 고등교육학교_학과수
FROM higher_education.panel_0101 WHERE 개방ID=? ORDER BY 조사년도
""", [open_id]).fetchall()
department_rows = connection.execute("""
SELECT DISTINCT 조사년도, 주야간계절구분명, 학과명, 학과상태명
FROM university_disclosure.panel_1017 WHERE 개방ID=? ORDER BY 조사년도
""", [open_id]).fetchall()
assert [row[0] for row in yearly_rows] == ['2022', '2023', '2024', '2025']
assert yearly_rows[0][4] == '2'
assert all(all(value == '0' for value in row[3:]) for row in yearly_rows[1:])
assert {row[1] for row in yearly_rows} == {'서울 종로구'}
assert {row[1] for row in department_rows} == {'야간'}
assert {row[2] for row in department_rows} == {'국제금융학과'}
assert {row[3] for row in department_rows} == {'폐과'}
yearly_rows, department_rows

([('2022', '서울 종로구', '본교', '0', '2', '0', '0', '0'),
  ('2023', '서울 종로구', '본교', '0', '0', '0', '0', '0'),
  ('2024', '서울 종로구', '본교', '0', '0', '0', '0', '0'),
  ('2025', '서울 종로구', '본교(제1캠퍼스)', '0', '0', '0', '0', '0')],
 [('2022', '야간', '국제금융학과', '폐과'),
  ('2023', '야간', '국제금융학과', '폐과'),
  ('2024', '야간', '국제금융학과', '폐과'),
  ('2025', '야간', '국제금융학과', '폐과')])

### 3. KEDI 원본의 폐교 상태


In [6]:
xlsx_ns = {'m': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}

def column_index(cell_reference):
    letters = re.match(r'[A-Z]+', cell_reference).group(0)
    result = 0
    for letter in letters:
        result = result * 26 + ord(letter) - 64
    return result - 1

def find_kedi_row(path, school_name):
    with zipfile.ZipFile(path) as archive:
        shared_root = ET.fromstring(archive.read('xl/sharedStrings.xml'))
        shared = [''.join(node.text or '' for node in item.findall('.//m:t', xlsx_ns))
                  for item in shared_root.findall('m:si', xlsx_ns)]
        sheet_root = ET.fromstring(archive.read('xl/worksheets/sheet1.xml'))
    for row in sheet_root.findall('.//m:row', xlsx_ns):
        values = {}
        for cell in row.findall('m:c', xlsx_ns):
            value_node = cell.find('m:v', xlsx_ns)
            if value_node is None:
                continue
            value = shared[int(value_node.text)] if cell.get('t') == 's' else value_node.text
            values[column_index(cell.get('r'))] = value
        if school_name in values.values():
            return [values.get(index, '') for index in range(max(values) + 1)]
    return None

school_name = '성균관대학교 무역대학원'
kedi_rows = {
    str(year): find_kedi_row(kedi_dir / f'{year}_kedi_higher_education_school.xlsx', school_name)
    for year in range(2022, 2025)
}
assert all(kedi_rows.values())
assert all(row[3] == school_name for row in kedi_rows.values())
assert all(row[4] == '폐교' for row in kedi_rows.values())
assert all(row[7] == '서울 종로구' for row in kedi_rows.values())
assert all(row[9] == '야간' for row in kedi_rows.values())
[(year, row[3], row[4], row[7], row[9]) for year, row in kedi_rows.items()]

[('2022', '성균관대학교 무역대학원', '폐교', '서울 종로구', '야간'),
 ('2023', '성균관대학교 무역대학원', '폐교', '서울 종로구', '야간'),
 ('2024', '성균관대학교 무역대학원', '폐교', '서울 종로구', '야간')]

## Takeaways

- `8463049173`는 **성균관대학교 무역대학원** 계열 폐교·폐과 잔존 ID로 분류하는 것이 적절하다.
- 2025년 9개 패널·49행의 실질 측정값은 모두 0이며 국제금융학과는 폐과다.
- 2022년 0101의 유일한 비영 실적은 졸업생 2명이고, 같은 해 KEDI에서 폐교 상태의 성균관대학교 무역대학원이 졸업생 기준 후보로 잡힌다.
- KEDI 2022~2024 원본에는 동일 학교가 폐교 상태로 반복되며, 2025 원본에서는 더 이상 학교 행이 확인되지 않는다.
- 직접 학교명 키가 없으므로 원본 ID를 보존하고 현행 기관 분석에서 제외하며 자동 조인은 하지 않는 것이 안전하다.